## Trabajo práctico 3

### Autor: Emmanuel Guerreiro - 47262

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin


In [2]:
TARGET_COLUMN = 'is_increment_24hs'

In [3]:
import pandas as pd

raw_df = pd.read_csv('data.csv')
raw_df.head()

,date,text,favorites,retweets,Toxic,Insult,Profanity,Derogatory,Sexual,"Death, Harm & Tragedy",...,Illicit Drugs,War & Conflict,Politics,Finance,Legal,btc_tweet_day,btc_24h_after,btc_48h_after,btc_delta_24h,btc_delta_48h
0,2020-03-03 01:34:00,I was thrilled to be back in the Great city of...,73748,17404,0.014622,0.010394,0.002904,0.003574,0.002435,0.059701,...,0.036585,0.472222,0.867769,0.079710,0.435185,8753.01,8700.00,9085.48,-53.01,332.47
1,2020-01-17 03:22:00,RT @CBS_Herridge: READ: Letter to surveillance...,0,7396,0.050346,0.050961,0.023985,0.008934,0.010127,0.085399,...,0.294118,0.350000,0.680934,0.081967,0.948357,8850.83,8903.26,8631.95,52.43,-218.88
2,2020-09-12 20:10:00,The Unsolicited Mail In Ballot Scam is a major...,80527,23502,0.258527,0.112138,0.058611,0.043741,0.015008,0.082609,...,0.700000,0.058824,0.937500,0.515901,0.858108,10359.99,10280.14,10698.03,-79.85,338.04
3,2020-01-17 13:13:00,RT @MZHemingway: Very friendly telling of even...,0,9081,0.018771,0.012035,0.004276,0.003942,0.002470,0.207207,...,0.057377,0.375000,0.912500,0.079710,0.333333,8850.83,8903.26,8631.95,52.43,-218.88
4,2020-01-17 00:11:00,RT @WhiteHouse: President @realDonaldTrump ann...,0,25048,0.016713,0.011194,0.004276,0.005187,0.002435,0.073482,...,0.057377,0.141892,0.969512,0.102302,0.948357,8850.83,8903.26,8631.95,52.43,-218.88


In [4]:
raw_df.columns
print(raw_df["date"].dtypes)
print(raw_df.shape)

object
(22451, 25)


In [5]:
class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop=None):
        self.columns_to_drop = columns_to_drop or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop)

In [6]:
class GenerateTarget(BaseEstimator, TransformerMixin):
    """
    Transformer that generates a binary target column (`target_movement`)
    from the numeric feature `btc_delta_24h_pct`.
    
    Returns the same DataFrame with the new column appended.
    """

    def fit(self, X, y=None):
        
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        
        X = X.copy()

        X[TARGET_COLUMN] = (X['btc_delta_24h'] > 0).astype(int)

        return X


In [7]:
import re
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class NormalizeData(BaseEstimator, TransformerMixin):
    """Transformer that normalizes column names to snake_case."""

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    @staticmethod
    def to_snake_case(name):
        """Convert a string to snake_case."""
        s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
        s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
        s3 = re.sub(r'[-\s]+', '_', s2)
        s4 = re.sub(r'_+', '_', s3)
        return s4.strip('_').lower()

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df.columns = [self.to_snake_case(col) for col in df.columns]
        return df


In [8]:
import numpy as np

class ManualFeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self
        

    def extract_hour(self, df):
        df["tmp_date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df["hour"] = df["tmp_date"].dt.hour
        df["normalized_hour"] = (df["hour"] - 6) % 24
        df.drop(columns=["tmp_date"], inplace=True)
        return df

    def extract_day_of_week(self, df):
        date_parsed = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df["day_name"] = date_parsed.dt.day_name().astype("category")
        df["day_of_week"] = date_parsed.dt.dayofweek
        df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
        return df

    def transform(self, df):
        
        df['btc_delta_24h_pct'] = df['btc_delta_24h'] / df['btc_tweet_day']
        df['btc_delta_48h_pct'] = df['btc_delta_48h'] / df['btc_tweet_day']


        df = self.extract_hour(df)
        df = self.extract_day_of_week(df)

        # Feature que junta otras de google
        
        # Otra feature mas

        return df

In [ ]:
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline, FeatureUnion

numeric_pipeline = FeatureUnion([
    ("scaled", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler())
    ])),
    ("pca", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MinMaxScaler()),
        ("pca", PCA(n_components=2))
    ]))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

column_transformer = ColumnTransformer([
    ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', categorical_pipeline, make_column_selector(dtype_exclude=np.number))
])

# === Global preprocessing pipeline ===
preprocessor = Pipeline([
    ("feature_engineering", ManualFeatureEngineering()),
    ("drop_columns", DropColumns(columns_to_drop=["date","text",
        'btc_24h_after', 'btc_48h_after', 'btc_delta_24h', 'btc_delta_48h', "btc_delta_48h_pct", "btc_delta_24h_pct"
    ])),
    ("column_transformer", column_transformer),
])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

data_prep_pipeline = Pipeline([
    ("generate_target", GenerateTarget()),
])

df_prepared = data_prep_pipeline.fit_transform(raw_df)

y = df_prepared[TARGET_COLUMN]
X = df_prepared.drop(columns=[TARGET_COLUMN])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


is_increment_24hs
1    9606
0    8354
Name: count, dtype: int64

In [11]:
from sklearn.model_selection import KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)


In [12]:
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import GridSearchCV

# Pipeline
rf_cv_pca_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    # ("drop_columns", DropColumns(columns_to_drop=["date"])),
    ("model", RandomForestClassifier
    (
        class_weight='balanced',
        random_state=42
    ))
])

# Grilla de parámetros
param_grid_rf = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    rf_cv_pca_pipeline,
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 10, ...], 'model__min_samples_split': [2, 5, ...], 'model__n_estimators': [50, 100, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,steps,"[('feature_engineering', ...), ('drop_columns', ...), ...]"


In [13]:
# Get the fitted preprocessor from the best estimator
preprocessor = grid_rf.best_estimator_.named_steps["preprocessing"]

# Transform your training data
X_processed = preprocessor.transform(X_train)

print("✅ Processed shape:", X_processed.shape)
ct = preprocessor.named_steps["column_transformer"]
feature_names = ct.get_feature_names_out()

print("🔹 Total features:", len(feature_names))
print("🔹 First 20 feature names:", feature_names[:100])

✅ Processed shape: (17960, 32)
🔹 Total features: 32
🔹 First 20 feature names: ['num__scaled__favorites' 'num__scaled__retweets' 'num__scaled__Toxic'
 'num__scaled__Insult' 'num__scaled__Profanity' 'num__scaled__Derogatory'
 'num__scaled__Sexual' 'num__scaled__Death, Harm & Tragedy'
 'num__scaled__Violent' 'num__scaled__Firearms & Weapons'
 'num__scaled__Public Safety' 'num__scaled__Health'
 'num__scaled__Religion & Belief' 'num__scaled__Illicit Drugs'
 'num__scaled__War & Conflict' 'num__scaled__Politics'
 'num__scaled__Finance' 'num__scaled__Legal' 'num__scaled__btc_tweet_day'
 'num__scaled__hour' 'num__scaled__normalized_hour'
 'num__scaled__day_of_week' 'num__scaled__is_weekend' 'num__pca__pca0'
 'num__pca__pca1' 'cat__day_name_Friday' 'cat__day_name_Monday'
 'cat__day_name_Saturday' 'cat__day_name_Sunday' 'cat__day_name_Thursday'
 'cat__day_name_Tuesday' 'cat__day_name_Wednesday']


In [14]:
from matplotlib.pyplot import figure, title
from sklearn.metrics import classification_report, confusion_matrix
from pandas import Series

print("Best parameters (Random Forest):")
print(grid_rf.best_params_)

def evaluar_modelo(nombre, y_true, y_pred):
    print(f"\n🔎 {nombre}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, digits=3))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

y_pred_logreg_grid = grid_rf.predict(X_test)

evaluar_modelo("Random Forest", y_test, y_pred_logreg_grid)



Best parameters (Random Forest):
{'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}

🔎 Random Forest

Classification report:
              precision    recall  f1-score   support

           0      0.757     0.646     0.697      2089
           1      0.727     0.820     0.770      2402

    accuracy                          0.739      4491
   macro avg      0.742     0.733     0.734      4491
weighted avg      0.741     0.739     0.736      4491

Confusion matrix:
[[1349  740]
 [ 433 1969]]


## XGBoost 

In [24]:
# XGBoost requires values encoded as a map with numpy numbers
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))


Class mapping: {np.int64(0): np.int64(0), np.int64(1): np.int64(1)}


In [28]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


pipeline_xgb = ImbPipeline([
    ("feature_engineer", ManualFeatureEngineering()),
    ("drop_columns", DropColumns(columns_to_drop=["date","text",
        'btc_24h_after', 'btc_48h_after', 'btc_delta_24h', 'btc_delta_48h', "btc_delta_48h_pct", "btc_delta_24h_pct"
    ])),
    ("column_transformer", column_transformer),
    ("model", xgb_model)
])

param_grid_xgb = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

grid_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid=param_grid_xgb,
    scoring="f1",
    cv=cv,
    n_jobs=-1
)


grid_xgb.fit(X_train, y_train_encoded)

y_pred_xgb_grid = grid_xgb.predict(X_test)
evaluar_modelo("XGBoost", y_test_encoded, y_pred_xgb_grid)


🔎 XGBoost

Classification report:
              precision    recall  f1-score   support

           0      0.904     0.891     0.898      2089
           1      0.906     0.918     0.912      2402

    accuracy                          0.905      4491
   macro avg      0.905     0.904     0.905      4491
weighted avg      0.905     0.905     0.905      4491

Confusion matrix:
[[1861  228]
 [ 197 2205]]
